# 07 — Interval Estimation
**References:** Casella & Berger (2002) Ch. 9 · Wasserman (2004) Ch. 6 · Lehmann & Romano (2005) Ch. 3

## Narrative thread
```
Pivot -> Exact CIs (t, chi-sq) -> Large-sample CIs -> Profile likelihood CIs -> Prediction intervals
```

## 1. Definition and interpretation

A **$(1-\alpha)$ confidence interval** $[L(\mathbf{X}), U(\mathbf{X})]$ satisfies:
$$P_\theta(L(\mathbf{X}) \leq \theta \leq U(\mathbf{X})) \geq 1 - \alpha \quad \forall\,\theta$$

> **Frequentist interpretation:** If we repeat the experiment many times, at least $(1-\alpha)\times 100\%$ of the resulting intervals will contain the true $\theta$. The interval is random; $\theta$ is fixed.

**Pivotal quantity:** A function $Q(\mathbf{X}, \theta)$ whose distribution does not depend on $\theta$.
If $Q \sim F$ (known), then $P(q_\alpha \leq Q \leq q_{1-\alpha}) = 1 - \alpha$ and solving for $\theta$ gives the CI.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

# ── Coverage simulation: frequentist interpretation of CIs ─────────────────
mu_true   = 5.0
sigma     = 2.0
n         = 20
alpha     = 0.05
n_trials  = 200

t_crit = stats.t.ppf(1 - alpha/2, df=n-1)
lowers, uppers, covers = [], [], []

for _ in range(n_trials):
    x   = np.random.normal(mu_true, sigma, n)
    xb  = x.mean()
    se  = x.std(ddof=1) / np.sqrt(n)
    lo  = xb - t_crit * se
    hi  = xb + t_crit * se
    lowers.append(lo); uppers.append(hi)
    covers.append(lo <= mu_true <= hi)

lowers, uppers, covers = map(np.array, [lowers, uppers, covers])
coverage_rate = covers.mean()

fig, ax = plt.subplots(figsize=(13, 6))
for i, (lo, hi, cover) in enumerate(zip(lowers, uppers, covers)):
    color = '#4361ee' if cover else '#f72585'
    ax.plot([lo, hi], [i, i], color=color, lw=1.2, alpha=0.7)
ax.axvline(mu_true, color='k', lw=2, linestyle='--', label=f'True $\\mu={mu_true}$')
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='#4361ee', label=f'Contains $\\mu$ ({covers.sum()}/{n_trials})'),
    Patch(color='#f72585', label=f'Misses $\\mu$ ({(~covers).sum()}/{n_trials})'),
    plt.Line2D([0],[0], color='k', lw=2, linestyle='--', label=f'True $\\mu={mu_true}$'),
])
ax.set_xlabel('Parameter value'); ax.set_ylabel('Trial')
ax.set_title(f'95% t-interval coverage: {coverage_rate*100:.1f}% of {n_trials} intervals contain true μ\n'
             f'(Expected: 95%). The interval is random; μ is fixed.')
plt.tight_layout()
plt.show()

## 2. Exact confidence intervals

### Mean of Normal, $\sigma^2$ unknown — Student's t
$$T = \frac{\bar{X} - \mu}{S/\sqrt{n}} \sim t(n-1) \quad \Rightarrow \quad \bar{X} \pm t_{n-1,\alpha/2} \cdot \frac{S}{\sqrt{n}}$$

### Variance of Normal — chi-squared pivot
$$\frac{(n-1)S^2}{\sigma^2} \sim \chi^2(n-1) \quad \Rightarrow \quad \left[\frac{(n-1)S^2}{\chi^2_{n-1,1-\alpha/2}},\; \frac{(n-1)S^2}{\chi^2_{n-1,\alpha/2}}\right]$$

### Proportion — Clopper-Pearson (exact Binomial)
Based on inverting the exact binomial test. For observed $x$ successes out of $n$:
$$L = B_{\alpha/2}(x, n-x+1) \qquad U = B_{1-\alpha/2}(x+1, n-x)$$
where $B_p(a,b)$ is the $p$-th quantile of Beta$(a,b)$.

In [ ]:
# ── Exact CIs vs approximations ───────────────────────────────────────────
from scipy.stats import beta as beta_dist

n      = 30
alpha  = 0.05

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# t-interval vs z-interval
sigma_known = 2.0
x_demo = np.random.normal(5, sigma_known, n)
xb, s  = x_demo.mean(), x_demo.std(ddof=1)

ax = axes[0]
methods = [
    ('t-interval (exact)', xb - stats.t.ppf(0.975, n-1)*s/np.sqrt(n), xb + stats.t.ppf(0.975, n-1)*s/np.sqrt(n), '#4361ee'),
    ('z-interval (sigma known)', xb - 1.96*sigma_known/np.sqrt(n), xb + 1.96*sigma_known/np.sqrt(n), '#f72585'),
]
for i, (name, lo, hi, c) in enumerate(methods):
    ax.barh(i, hi-lo, left=lo, color=c, alpha=0.7, height=0.4)
    ax.text((lo+hi)/2, i, f'[{lo:.2f}, {hi:.2f}]', ha='center', va='center', fontsize=8, color='white', fontweight='bold')
ax.axvline(5, color='k', lw=1.5, linestyle='--', label='True μ=5')
ax.set_yticks([0, 1]); ax.set_yticklabels([m[0] for m in methods])
ax.set_title('t vs z interval for mean\n(t wider: accounts for unknown σ)')
ax.legend(fontsize=8)

# Chi-squared CI for variance
ns_var = np.arange(5, 101)
chi_lo = [(n_-1) / stats.chi2.ppf(0.975, n_-1) for n_ in ns_var]
chi_hi = [(n_-1) / stats.chi2.ppf(0.025, n_-1) for n_ in ns_var]
ax2 = axes[1]
ax2.fill_between(ns_var, chi_lo, chi_hi, alpha=0.3, color='#4361ee')
ax2.plot(ns_var, chi_lo, color='#4361ee', lw=2, label='CI bounds (S²=true σ²)')
ax2.plot(ns_var, chi_hi, color='#4361ee', lw=2)
ax2.axhline(1, color='#f72585', lw=2, linestyle='--', label='True ratio=1')
ax2.set_xlabel('n'); ax2.set_ylabel('CI bounds (σ²_hat/σ²_true)')
ax2.set_title('χ² CI for variance: width vs n\nHighly asymmetric — CI not centered at S²')
ax2.legend(fontsize=8)

# Clopper-Pearson vs Wald CI for proportion
x_prop = 3   # successes out of n=30
p_hat  = x_prop / n
ps     = np.linspace(0, 0.35, 300)

cp_lo = beta_dist.ppf(alpha/2, x_prop, n - x_prop + 1)
cp_hi = beta_dist.ppf(1 - alpha/2, x_prop + 1, n - x_prop)
wald_lo = max(0, p_hat - 1.96 * np.sqrt(p_hat*(1-p_hat)/n))
wald_hi = p_hat + 1.96 * np.sqrt(p_hat*(1-p_hat)/n)

ax3 = axes[2]
ax3.barh(1, cp_hi-cp_lo,   left=cp_lo,   color='#4361ee', alpha=0.7, height=0.3, label=f'Clopper-Pearson [{cp_lo:.3f}, {cp_hi:.3f}]')
ax3.barh(0, wald_hi-wald_lo, left=wald_lo, color='#f72585', alpha=0.7, height=0.3, label=f'Wald [{wald_lo:.3f}, {wald_hi:.3f}]')
ax3.axvline(p_hat, color='k', lw=1.5, linestyle='--', label=f'$\\hat p = {p_hat:.2f}$')
ax3.set_yticks([0, 1]); ax3.set_yticklabels(['Wald (approx)', 'Clopper-Pearson (exact)'])
ax3.set_title(f'Proportion CI: x={x_prop}, n={n}\nExact CP guarantees coverage; Wald does not for small n')
ax3.legend(fontsize=7)

plt.suptitle('Exact vs Approximate Confidence Intervals', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Large-sample confidence intervals

From the asymptotic normality of the MLE (notebook 06):
$$\hat{\theta} \pm z_{\alpha/2} \cdot \widehat{\text{SE}}(\hat{\theta}), \qquad \widehat{\text{SE}} = 1/\sqrt{\hat{I}_n}$$

**Likelihood ratio CI** (more accurate than Wald for small $n$):
$$\{\theta : 2[\ell(\hat{\theta}) - \ell(\theta)] \leq \chi^2_{1,1-\alpha}\}$$

By Wilks' theorem, $2[\ell(\hat\theta) - \ell(\theta_0)] \xrightarrow{d} \chi^2(1)$ under $H_0: \theta = \theta_0$.

**Three asymptotically equivalent test statistics:**

| Name | Formula | Notes |
|---|---|---|
| Wald | $(\hat\theta - \theta_0)^2 \hat I_n$ | Simplest; needs Hessian |
| Score (Rao) | $[\ell'(\theta_0)]^2 / I_n(\theta_0)$ | No optimization needed |
| Likelihood ratio | $2[\ell(\hat\theta) - \ell(\theta_0)]$ | Best finite-sample behavior |

## 4. Key takeaways

| Interval type | When to use | Key formula |
|---|---|---|
| t-interval | Normal data, σ unknown | $\bar X \pm t_{n-1}\cdot S/\sqrt{n}$ |
| χ² interval | Normal variance | Invert $\chi^2(n-1)$ pivot |
| Clopper-Pearson | Proportion, any $n$ | Beta quantiles |
| Wald (MLE-based) | Large $n$, any model | $\hat\theta \pm z\cdot\hat{\text{SE}}$ |
| LR interval | Better coverage than Wald | Invert Wilks' test |

**Next:** notebook 08 — hypothesis testing theory: Neyman-Pearson lemma, UMP tests, power.